# Scikit Learn MLP - it has only EDA PPG


In [1]:
import os
os.chdir("/notebooks")

In [2]:
from lib.install import prepare_env
_ = prepare_env()


Installed versions:
tensorflow: 2.15.0
tensorflow-probability: missing
tf-keras: missing
transformers: 4.35.2
protobuf: 4.23.4
datasets: 2.14.5
evaluate: missing
scikit-learn: 1.3.0
imbalanced-learn: missing
wordcloud: missing
Version mismatch: tensorflow installed=2.15.0, required=2.18.0
Missing: tensorflow-probability
Missing: tf-keras
Version mismatch: transformers installed=4.35.2, required=4.46.3
Version mismatch: protobuf installed=4.23.4, required=4.25.8
Missing: evaluate
Missing: imbalanced-learn
Missing: wordcloud

Reinstalling required packages...

Installed versions:
tensorflow: 2.18.0
tensorflow-probability: 0.25.0
tf-keras: 2.18.0
transformers: 4.46.3
protobuf: 4.25.8
datasets: 5.0.0
evaluate: 0.4.6
scikit-learn: 1.9.0
imbalanced-learn: 0.14.2
wordcloud: 1.9.6

Done — restart the kernel before importing TensorFlow or Transformers.
After restart, run: check_tensorflow_cuda()


#### 1. Import

In [3]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
import tensorflow as tf
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.neighbors import NearestNeighbors
from lib.logger import Logger

#### 2. Config

In [4]:
hidden_layers = (100, 50)
latent_dim    = hidden_layers[1]
batch_size    = 200
epochs        = 200
seed_values   = [42, 43, 111]
lr            = 1e-3
TASK          = os.getenv("TASK", "AROUSAL")
MODALITY      = os.getenv("MODALITY", "EDA")
EXPERIMENT    = os.getenv("EXPERIMENT", "SIMPLE")
MODEL_NAME    = "MLP"

FSCORE_CALCULATION = 'binary'

DATA_CONFIG = {
    "eda_path": "EEVR/Data_files/eda.csv",
    "ppg_path": "EEVR/Data_files/ppg.csv",
}

#### 3. Utility Functions

In [5]:
# ============================================================
# Utility: CSV read
# ============================================================
def csv_read(path):
    df = pd.read_csv(path)
    return df

logger = Logger(MODEL_NAME,MODALITY,TASK,EXPERIMENT,threshold=0.5)

# ============================================================
# Utility: seed everything
# ============================================================
def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    return seed

# ============================================================
# Utility: Feature Analysis
# ============================================================
def feature_analysis(df, relevant_features, identifiers):
    df_features = df[relevant_features]
    arousals = df["arousal_category"].tolist()
    num_high_arousal = arousals.count(1)
    num_low_arousal = arousals.count(0)
    logger.print(
        f"The Number of entries of High Arousal: {num_high_arousal} "
        f"& Low Arousal: {num_low_arousal}"
    )
    valence = df["valence_category"].tolist()
    num_high_valence = valence.count(1)
    num_low_valence = valence.count(0)

    logger.print(
        f"The Number of entries of High Valence: {num_high_valence} "
        f"& Low Valence: {num_low_valence}"
    )
    taskwiselabel_list = df["taskwiselabel"].tolist()
    num_zeros = taskwiselabel_list.count(0)
    num_ones = taskwiselabel_list.count(1)

    logger.print(
        f"The Number of entries of Positive Task: {num_ones} "
        f"& Negative Task: {num_zeros}"
    )
    pi = df["Participant ID"]
    vi = df["Video ID"]
    df_required = df[relevant_features + identifiers]
    return df_required, pi, vi


# ============================================================
# Utility: metrics
# ============================================================
def compute_metrics(y_true, y_pred, n_classes):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    accuracy = accuracy_score(y_true, y_pred)
    
    if FSCORE_CALCULATION.lower() == "weighted" or n_classes > 2:
        f1score = f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        )
    else:
        f1score = f1_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

    return {
        "acc": accuracy,
        "f1": f1score,
    }

# ============================================================
# Utility: balance data SIMPLE EEVR
# ============================================================
def balance_train_data_simple(X_train, y_train):
    """
    Duplicate the minority class once and concatenate it back.
    No shuffling is applied.
    """

    X_train = X_train.reset_index(drop=True)                         # reset feature index
    y_train = y_train.reset_index(drop=True)                         # reset label index

    counts = y_train.value_counts()                                  # class counts
    minority_class = counts.idxmin()                                 # minority class label

    minority_idx = y_train[y_train == minority_class].index           # minority row indices

    X_minority = X_train.loc[minority_idx]                            # minority feature rows
    y_minority = y_train.loc[minority_idx]                            # minority labels

    X_balanced = pd.concat(
        [X_train, X_minority],
        axis=0,
        ignore_index=True,
    )                                                                # add minority rows at the end

    y_balanced = pd.concat(
        [y_train, y_minority],
        axis=0,
        ignore_index=True,
    )                                                                # add minority labels at the end

    return X_balanced, y_balanced


# ============================================================
# Utility: balance data SMOTE
# ============================================================
def balance_train_data_smote(X_train, y_train, random_state=42):
    """
    Balance X_train and y_train using SMOTE.
    If the minority class has only one sample, RandomOverSampler is used instead.
    """

    # Reset index to avoid alignment issues
    X_train = X_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    # Check class counts
    counts = y_train.value_counts()
    minority_count = counts.min()

    # Use RandomOverSampler if SMOTE cannot be applied
    if minority_count <= 1:
        sampler = RandomOverSampler(random_state=random_state)
    else:
        sampler = SMOTE(
            random_state=random_state,
            k_neighbors=min(5, minority_count - 1)
        )

    X_balanced, y_balanced = sampler.fit_resample(X_train, y_train)

    # Convert back to pandas format
    X_balanced = pd.DataFrame(X_balanced, columns=X_train.columns)
    y_balanced = pd.Series(y_balanced, name=y_train.name)

    return X_balanced, y_balanced

#### 4. Read Data

In [6]:
identifiers = [
    "Participant ID",
    "Video ID",
    "arousal_category",
    "valence_category",
    "taskwiselabel"
]


if MODALITY == 'EDA':
    features = [
        "ku_eda", "sk_eda", "dynrange", "slope", "variance",
        "entropy", "insc", "fd_mean", "max_scr", "min_scr",
        "nSCR", "meanAmpSCR", "meanRespSCR", "sumAmpSCR", "sumRespSCR",
    ]
    df = csv_read(DATA_CONFIG["eda_path"])
    df_required, df_pi, df_vi = feature_analysis(
        df,
        features,
        identifiers,
    )
elif MODALITY == 'PPG':
    features = ["BPM", "IBI", "PPG_Rate_Mean", "HRV_MedianNN", "HRV_Prc20NN", "HRV_MinNN", "HRV_HTI", "HRV_TINN", "HRV_LF",
                "HRV_VHF", "HRV_LFn", "HRV_HFn", "HRV_LnHF", "HRV_SD1SD2", "HRV_CVI", "HRV_PSS", "HRV_PAS", "HRV_PI",
                "HRV_C1d", "HRV_C1a", "HRV_DFA_alpha1", "HRV_MFDFA_alpha1_Width", "HRV_MFDFA_alpha1_Peak", "HRV_MFDFA_alpha1_Mean",
                "HRV_MFDFA_alpha1_Max", "HRV_MFDFA_alpha1_Delta", "HRV_MFDFA_alpha1_Asymmetry", "HRV_ApEn", "HRV_ShanEn",
                "HRV_FuzzyEn", "HRV_MSEn", "HRV_CMSEn", "HRV_RCMSEn", "HRV_CD", "HRV_HFD", "HRV_KFD", "HRV_LZC"]

    df = csv_read(DATA_CONFIG["ppg_path"])
    df_required, df_pi, df_vi = feature_analysis(
        df,
        features,
        identifiers,
    )
else:
    raise Error(f'Invalid Modality {MODALITY}')


if TASK == 'AROUSAL':
    label_column = 'arousal_category'
    n_class = 2
elif TASK == 'VALENCE':
    label_column = 'valence_category'
    n_class = 2
elif TASK == 'STIMULUS-LABEL':
    label_column = 'taskwiselabel'
    n_class = 2
else:
    raise Error(f'Invalid task {TASK}')

The Number of entries of High Arousal: 114 & Low Arousal: 219
The Number of entries of High Valence: 202 & Low Valence: 131
The Number of entries of Positive Task: 148 & Negative Task: 185


In [7]:
X = df_required[features]
y = df_required[label_column]
participant_id = df_pi
video_id       = df_vi

### 5. Run LOSO Validation


In [ ]:
performance_rows = []

for seed in seed_values:
    set_seed(seed)                                                      # repeat experiment for each seed
    loso = LeaveOneGroupOut()                                           # LOSO split by participant

    for fold_no, (train_idx, test_idx) in enumerate(loso.split(X, y, groups=participant_id), start=1):
        test_pid = participant_id.iloc[test_idx].iloc[0]                # held-out participant

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]           # train/test features
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]           # train/test labels

        sample_id = participant_id.iloc[test_idx].astype(str).values + "_" + video_id.iloc[test_idx].astype(str).values  # participant_video id
        
        if EXPERIMENT=="SMOTE":
            X_train,y_train = balance_train_data_smote(X_train,y_train) # balance train data using SMOTE
        elif EXPERIMENT=="SIMPLE":
            X_train,y_train = balance_train_data_simple(X_train,y_train) # balance train data by duplication
            
        model = MLPClassifier(random_state=seed, hidden_layer_sizes=hidden_layers, max_iter=epochs, learning_rate_init=lr) # MLP classifier
        model.fit(X_train, y_train)                                    # train model

        pred_proba = model.predict_proba(X_test)[:, 1]                 # P(class 1)
        y_pred = model.predict(X_test)                                 # predicted class

        y_test_arr = np.asarray(y_test).astype("int32")                # true labels as int array
        y_pred_arr = np.asarray(y_pred).astype("int32")                # predicted labels as int array

        logger.add_predictions(
            seed=seed,
            fold=fold_no,
            pid=test_pid,
            sample_id=sample_id,
            y_true=y_test_arr,
            pred_proba=pred_proba,
        )                                                              # save gt, pred_proba, pred_class_0.5

        metrics = compute_metrics(y_test_arr, y_pred_arr, n_class)      # compute accuracy and F1

        performance_rows.append({"seed": seed, "pid": test_pid, "acc": metrics["acc"], "f1": metrics["f1"]}) # store fold result

        perf_df = pd.DataFrame(performance_rows)                       # all fold results so far
        seed_mean_df = perf_df.groupby("seed")[["acc", "f1"]].mean().reset_index() # mean over participants per seed
        global_mean = seed_mean_df[["acc", "f1"]].mean()               # mean over seeds
        global_std = seed_mean_df[["acc", "f1"]].std()                 # std over seeds

        logger.print(f"Running Mean | Seed={seed} | pid={test_pid} | Acc={global_mean['acc']:.4f} | F1={global_mean['f1']:.4f}")

summary_df = pd.DataFrame([{"seed": "Mean ± Std", "acc": f"{global_mean['acc']:.4f} ± {global_std['acc']:.4f}", "f1": f"{global_mean['f1']:.4f} ± {global_std['f1']:.4f}"}])
seed_summary_df = pd.concat([seed_mean_df, summary_df], axis=0, ignore_index=True)

logger.print("Seed-wise mean results:")
logger.print(seed_summary_df)

logger.save_npz()                                                      # save sample_id, gt, pred_proba, pred_class_0.5, threshold, seed, fold, pid

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Running Mean | Seed=42 | pid=1 | Acc=0.5556 | F1=0.0000


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Running Mean | Seed=42 | pid=2 | Acc=0.5000 | F1=0.3077


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


...

Running Mean | Seed=42 | pid=27 | Acc=0.5467 | F1=0.3739


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Running Mean | Seed=42 | pid=28 | Acc=0.5641 | F1=0.3595


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Running Mean | Seed=42 | pid=29 | Acc=0.5597 | F1=0.3462
